In [1]:
import pandas as pd
import os
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

root_path = os.path.dirname((os.getcwd()))

In [15]:
data_path = os.path.join(Path(os.getcwd()).resolve().parent, "dataset.xlsx")
df = pd.read_excel(data_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12095 entries, 0 to 12094
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   studentID         12095 non-null  object 
 1   classID           12095 non-null  object 
 2   timeStamp         12095 non-null  object 
 3   studentEmotion    12095 non-null  object 
 4   finalScore        12095 non-null  float64
 5   totalImages       12095 non-null  int64  
 6   learningTimes     12095 non-null  int64  
 7   finishedLession   12095 non-null  int64  
 8   avgTimeLearn      12095 non-null  float64
 9   avgTimeFinish     12095 non-null  float64
 10  percentageFinish  12095 non-null  float64
 11  timeInWeek        12095 non-null  float64
 12  inTime            12095 non-null  float64
 13  outTime           12095 non-null  float64
 14  inWeekday         12095 non-null  float64
 15  outWeekday        12095 non-null  float64
dtypes: float64(9), int64(3), object(4)
memor

In [16]:
df = df.sort_values(by=["studentID", "timeStamp"])

In [17]:
# group emotions into a "document" per student
docs = (
    df.groupby("studentID")["studentEmotion"]
      .apply(lambda x: " ".join(x.astype(str)))
)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",  # keep single-word emotions
    lowercase=False
)

tfidf_matrix = vectorizer.fit_transform(docs)

In [18]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=docs.index,
    columns=vectorizer.get_feature_names_out()
)

In [19]:
tfidf_df

,happy,neutral,sad
studentID,,,
student_001,0.092949,0.841106,0.532824
student_002,0.000000,0.990096,0.140389
student_003,0.012045,0.399899,0.916480
student_004,0.066574,0.910152,0.408889
student_005,0.034871,0.976173,0.214175
...,...,...,...
student_165,0.000000,1.000000,0.000000
student_166,0.000000,1.000000,0.000000
student_167,0.438892,0.000000,0.898540


In [20]:
df = df.drop_duplicates(subset=["studentID"])

In [21]:
df_final = df[["studentID", "finalScore"]].merge(tfidf_df, on="studentID", how="inner")

In [22]:
print(df_final.shape)
df_final.head(3)

(169, 5)


,studentID,finalScore,happy,neutral,sad
0,student_001,10.0,0.092949,0.841106,0.532824
1,student_002,7.6,0.000000,0.990096,0.140389
2,student_003,8.4,0.012045,0.399899,0.916480


In [23]:
def log_transform(df, columns, eps=1e-6):
    df = df.copy()
    
    for col in columns:
        df[col] = np.log1p(np.clip(df[col], -1 + eps, None))
    
    return df

def z_score_transform(df, id_col):
    df = df.copy()
    cols_to_scale = df.columns.difference([id_col])
    scaler = StandardScaler()
    df.loc[:, cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
    return df

In [24]:
cols = df_final.drop(["studentID", "finalScore"], axis=1).columns
master_df = log_transform(df=df_final, columns=cols)
master_df = z_score_transform(df=master_df, id_col="studentID")

In [25]:
master_df.head(5)

,studentID,finalScore,happy,neutral,sad
0,student_001,0.743599,0.013875,0.446626,0.247295
1,student_002,-0.318012,-0.628991,0.828138,-1.013355
2,student_003,0.035858,-0.542394,-0.896551,1.199482
3,student_004,-0.141077,-0.162813,0.627127,-0.112091
4,student_005,-1.025754,-0.381066,0.793716,-0.746105


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [26]:
target_col = "finalScore"
X = master_df.drop(columns=["studentID", "finalScore"], axis=1)
y = master_df[target_col]

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = ElasticNet(alpha=0.05, l1_ratio=0.5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [56]:
def eval_reg(y_true, y_pred, name=""):
    print(f"\n{name}")
    print("MSE:", mean_squared_error(y_true, y_pred))
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("R2 :", r2_score(y_true, y_pred))

eval_reg(y_test, model.predict(X_test), "Test")


Test
MSE: 0.9420011563069578
MAE: 0.5828364516262421
R2 : 0.053302307909536495
